# Chapter 10 — Gas Processing and Glycols: Scientific Figures

**Figures:**
1. TEG\u2013water VLE: CPA prediction at absorber conditions
2. Water dew point depression vs TEG purity and circulation rate
3. MEG\u2013water phase behavior and hydrate inhibition
4. TEG solver benchmark (from paperlab teg_cpa_solvers data)

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

In [3]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

## Figure 10.1 — TEG\u2013Water VLE at Absorber Conditions

In [4]:
# TEG-water VLE: activity coefficients and Pxy at 25°C
x_teg = np.arange(0.0, 1.01, 0.05)
P_vals = []
y_water = []

for xt in x_teg:
    xw = 1.0 - xt
    if xw < 1e-10: xw = 1e-10
    if xt < 1e-10: xt = 1e-10
    try:
        f = SystemSrkCPAstatoil(273.15 + 25.0, 1.0)
        f.addComponent("TEG", float(xt))
        f.addComponent("water", float(xw))
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.bubblePointPressureFlash(False)
        f.initProperties()
        P_vals.append(float(f.getPressure("bara")))
        y_w = float(f.getPhase("gas").getComponent("water").getx())
        y_water.append(y_w)
    except Exception:
        P_vals.append(np.nan)
        y_water.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(x_teg[:len(P_vals)], P_vals, color=BLUE, linewidth=1.5, label="CPA bubble curve")
ax.set_xlabel("$x_{\\rm TEG}$ (liquid mole fraction)")
ax.set_ylabel("Pressure (bar)")
ax.set_title("TEG\u2013Water Pxy at 25\u00b0C")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch10_01_teg_water_vle.png")

Saved: fig_ch10_01_teg_water_vle.png


## Figure 10.2 — Water Dew Point Depression with TEG

In [5]:
# Water dew point of gas contacted with TEG at various purities
teg_purities_wt = [98.0, 99.0, 99.5, 99.9]  # wt% TEG
colors_teg = [BLUE, ORANGE, GREEN, PURPLE]

# Estimate water content after contacting gas with TEG
fig, ax = plt.subplots(figsize=(3.5, 3.0))

P_contact = 70.0  # bar
T_contact = 30.0  # C

for purity, col in zip(teg_purities_wt, colors_teg):
    # TEG mass fraction
    w_teg = purity / 100.0
    w_water = 1.0 - w_teg
    # Approximate mole fractions (MW_TEG=150.17, MW_water=18.015)
    n_teg = w_teg / 150.17
    n_water = w_water / 18.015
    x_teg_mol = n_teg / (n_teg + n_water)
    x_water_mol = n_water / (n_teg + n_water)

    # Water dew point at different pressures
    P_range = np.arange(20, 160, 10)
    dew_temps = []
    for P in P_range:
        try:
            f = SystemSrkCPAstatoil(273.15 + 30.0, float(P))
            f.addComponent("methane", 0.90)
            f.addComponent("water", float(x_water_mol * 0.001))  # small amount
            f.addComponent("TEG", float(x_teg_mol * 0.001))
            f.setMixingRule(10)
            f.setMultiPhaseCheck(True)
            ops = ThermodynamicOperations(f)
            ops.waterDewPointTemperatureFlash()
            Tdew = float(f.getTemperature("C"))
            dew_temps.append(Tdew)
        except Exception:
            dew_temps.append(np.nan)
    ax.plot(P_range, dew_temps, color=col, linewidth=1.2,
            label=f"{purity:.1f} wt%")

ax.set_xlabel("Pressure (bar)")
ax.set_ylabel("Water dew point (\u00b0C)")
ax.set_title("Water dew point: gas + TEG")
ax.legend(title="TEG purity", frameon=True, fontsize=7, title_fontsize=7)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch10_02_dew_point_teg.png")

Saved: fig_ch10_02_dew_point_teg.png


## Figure 10.3 — MEG\u2013Water Activity Coefficients

In [6]:
# MEG-water: CPA activity coefficients vs composition
x_meg = np.arange(0.05, 0.96, 0.05)
gamma_meg, gamma_water = [], []

for xm in x_meg:
    xw = 1.0 - xm
    try:
        f = SystemSrkCPAstatoil(273.15 + 25.0, 1.0)
        f.addComponent("MEG", float(xm))
        f.addComponent("water", float(xw))
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        # Activity coefficients
        g_m = float(f.getPhase(0).getActivityCoefficient(0))
        g_w = float(f.getPhase(0).getActivityCoefficient(1))
        gamma_meg.append(g_m)
        gamma_water.append(g_w)
    except Exception:
        gamma_meg.append(np.nan)
        gamma_water.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(x_meg[:len(gamma_meg)], gamma_meg, color=BLUE, linewidth=1.5, label=r"$\gamma_{\rm MEG}$")
ax.plot(x_meg[:len(gamma_water)], gamma_water, color=ORANGE, linewidth=1.5, linestyle="--",
        label=r"$\gamma_{\rm water}$")
ax.axhline(y=1.0, color="grey", linestyle=":", alpha=0.5)
ax.set_xlabel("$x_{\\rm MEG}$")
ax.set_ylabel("Activity coefficient")
ax.set_title("MEG\u2013Water at 25\u00b0C (CPA)")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch10_03_meg_water_gamma.png")

Saved: fig_ch10_03_meg_water_gamma.png
